# Agente conversacional sencillo con OpenAI — Prueba Técnica

**Objetivo:** preguntar en lenguaje natural por precios históricos, relaciones estadísticas, estimaciones y presupuestos sin construir un sistema complejo.

**Diseño:** archivos CSV $\rightarrow$ cálculos sencillos con Pandas $\rightarrow$ contexto verificable $\rightarrow$ OpenAI API $\rightarrow$ respuesta. Sin Gradio, LangChain, bases vectoriales ni *function calling*.

> Este notebook funciona con los cuatro CSV de la prueba. Sin esos resultados utiliza exclusivamente una **referencia ingenua**: último precio mensual observado para el siguiente mes. No presenta esa referencia como pronóstico de un modelo avanzado.

**Límite temporal:** los datos de equipos terminan en 2023. Este prototipo **no** informa precios vigentes en 2026 ni realiza extrapolaciones a esa fecha.



## 1. Instalación y bibliotecas




In [1]:
%pip -q install openai pandas numpy

import os
from pathlib import Path
import pandas as pd
import numpy as np


## 2. Archivos de entrada

In [2]:
CARPETA = Path('/content') if Path('/content').exists() else Path('/mnt/data/datos_prueba')
if not (CARPETA / 'historico_equipos.csv').exists():
    try:
        from google.colab import files
        print('Sube historico_equipos.csv (y, si quieres, pronosticos.csv / metricas.csv).')
        files.upload()
        CARPETA = Path('/content')
    except ImportError:
        raise FileNotFoundError('Ubica historico_equipos.csv en la carpeta indicada por CARPETA.')

historico = pd.read_csv(CARPETA / 'historico_equipos.csv', parse_dates=['Date'])
COLUMNAS = ['Price_X','Price_Y','Price_Z','Price_Equipo1','Price_Equipo2']
for columna in COLUMNAS:
    historico[columna] = pd.to_numeric(historico[columna], errors='coerce')
historico = (historico.dropna(subset=['Date']+COLUMNAS)
             .drop_duplicates(subset='Date', keep='last')
             .sort_values('Date'))
historico = historico.loc[(historico[COLUMNAS] > 0).all(axis=1)].copy()
if historico.empty:
    raise ValueError('No quedaron registros válidos.')
mensual = (historico.set_index('Date')[COLUMNAS]
           .resample('ME').last().dropna())
print(f'Registros válidos: {len(historico):,}. Periodo: {historico.Date.min():%Y-%m-%d} a {historico.Date.max():%Y-%m-%d}.')
print('Último mes disponible:', mensual.index[-1].date())


Registros válidos: 3,530. Periodo: 2010-01-04 a 2023-08-31.
Último mes disponible: 2023-08-31


## 3. Resultados numéricos: estadísticas, correlaciones y pronósticos

**¿Qué mejora respecto a una consulta directa a GPT?:** los precios, correlaciones y presupuestos se calculan en Python; el modelo de lenguaje solo los explica.

- **Histórico:** último valor observado por serie.
- **Correlación:** asociación de Pearson en niveles de precio; no se interpreta como causalidad.
- **Pronóstico de referencia:** último precio mensual conocido para el siguiente mes; es un *baseline*, no el modelo Ridge o Random Forest del notebook de entrenamiento.
- **Archivos opcionales:** si existen y tienen las columnas esperadas, se utilizan pronósticos y métricas exportados de los modelos reales en lugar del baseline.

No pedimos al modelo generativo que invente predicciones.

In [3]:
objetivos = ['Price_Equipo1','Price_Equipo2']
insumos = ['Price_X','Price_Y','Price_Z']
correlaciones = mensual[insumos+objetivos].corr().loc[insumos,objetivos]
ultimo_mes = mensual.index[-1]
fecha_referencia = ultimo_mes + pd.offsets.MonthEnd(1)
pronosticos = pd.DataFrame({
    'Equipo': objetivos,
    'Fecha': [fecha_referencia.date().isoformat()]*2,
    'Prediccion': [float(mensual[e].iloc[-1]) for e in objetivos],
    'Metodo': ['Naive: último precio mensual']*2,
})
metricas = pd.DataFrame()

ruta_pronosticos = CARPETA / 'pronosticos.csv'
if ruta_pronosticos.exists():
    importado = pd.read_csv(ruta_pronosticos)
    requeridas = {'Equipo','Fecha','Prediccion'}
    if requeridas.issubset(importado.columns):
        importado['Fecha'] = pd.to_datetime(importado['Fecha']).dt.date.astype(str)
        importado['Prediccion'] = pd.to_numeric(importado['Prediccion'],errors='coerce')
        importado = importado.loc[importado['Equipo'].isin(objetivos) & importado['Prediccion'].notna()].copy()
        if not importado.empty:
            if 'Metodo' not in importado.columns:
                importado['Metodo'] = 'Modelo exportado (ver notebook de entrenamiento)'
            pronosticos = importado[['Equipo','Fecha','Prediccion','Metodo']]
    else:
        print('Aviso: pronosticos.csv no tiene las columnas requeridas. Se usa baseline.')
ruta_metricas = CARPETA / 'metricas.csv'
if ruta_metricas.exists():
    posible = pd.read_csv(ruta_metricas)
    if {'Equipo','Modelo','MAE'}.issubset(posible.columns):
        metricas = posible[['Equipo','Modelo','MAE']].copy()
    else:
        print('Aviso: metricas.csv sin columnas Equipo, Modelo, MAE. No se citan métricas.')

print('Correlaciones históricas (no causales):')
display(correlaciones.round(3))
print('Pronósticos disponibles:')
display(pronosticos)


Correlaciones históricas (no causales):


,Price_Equipo1,Price_Equipo2
Price_X,0.535,0.539
Price_Y,0.997,0.917
Price_Z,0.848,0.982


Pronósticos disponibles:


,Equipo,Fecha,Prediccion,Metodo
0,Price_Equipo1,2023-09-30,451.73,Naive: último precio mensual
1,Price_Equipo2,2023-09-30,955.35,Naive: último precio mensual


## 4. Construcción del contexto verificable

Esta función resume solamente información derivada de los archivos cargados. En cada pregunta también podemos incorporar un presupuesto calculado con cantidades explícitas.

**Importante:** cantidades como 15 o 10 son ejemplos, no pedidos reales. No sumar materias primas por separado a los equipos sin justificar el supuesto de compra.

In [5]:
import json

def generar_contexto(cantidades=None):
    ultima = mensual.iloc[-1]
    datos = {
        'alcance': 'Demostración académica, no precios actuales ni recomendación de compra',
        'periodo_historico_inicio': str(historico.Date.min().date()),
        'periodo_historico_fin': str(historico.Date.max().date()),
        'ultimo_mes_observado': str(ultimo_mes.date()),
        'ultimos_precios_observados': {c: round(float(ultima[c]),2) for c in COLUMNAS},
        'correlaciones_en_niveles_no_causales': {
            equipo: {materia: round(float(correlaciones.loc[materia,equipo]),3) for materia in insumos}
            for equipo in objetivos
        },
        'pronosticos_disponibles': pronosticos.to_dict('records'),
        'metricas_disponibles': metricas.to_dict('records'),
        'advertencia': 'No se dispone de precios actuales de 2026. Nunca inventar fechas, precios, resultados o niveles de confianza.'
    }
    if cantidades is not None:
        calculos = []
        for equipo,cantidad in cantidades.items():
            if equipo not in objetivos or not isinstance(cantidad,(int,float)) or cantidad < 0:
                raise ValueError('Cantidad o equipo inválido')
            filas = pronosticos.loc[pronosticos.Equipo.eq(equipo)]
            if len(filas) != 1:
                raise ValueError(f'Se requiere exactamente un pronóstico para {equipo}; elige una fecha y filtra antes.')
            fila = filas.iloc[0]
            calculos.append({'equipo':equipo,'cantidad_hipotetica':cantidad,
                             'fecha':fila['Fecha'],'precio_estimado':round(float(fila['Prediccion']),2),
                             'subtotal':round(float(fila['Prediccion'])*cantidad,2)})
        datos['presupuesto_hipotetico'] = {'detalle':calculos,'total':round(sum(x['subtotal'] for x in calculos),2)}
    return json.dumps(datos,ensure_ascii=False,indent=2,default=str)

contexto = generar_contexto(cantidades={'Price_Equipo1':15,'Price_Equipo2':10})
print(contexto[:1800] + ('...' if len(contexto)>1800 else ''))


{
  "alcance": "Demostración académica, no precios actuales ni recomendación de compra",
  "periodo_historico_inicio": "2010-01-04",
  "periodo_historico_fin": "2023-08-31",
  "ultimo_mes_observado": "2023-08-31",
  "ultimos_precios_observados": {
    "Price_X": 86.86,
    "Price_Y": 545.0,
    "Price_Z": 2165.25,
    "Price_Equipo1": 451.73,
    "Price_Equipo2": 955.35
  },
  "correlaciones_en_niveles_no_causales": {
    "Price_Equipo1": {
      "Price_X": 0.535,
      "Price_Y": 0.997,
      "Price_Z": 0.848
    },
    "Price_Equipo2": {
      "Price_X": 0.539,
      "Price_Y": 0.917,
      "Price_Z": 0.982
    }
  },
  "pronosticos_disponibles": [
    {
      "Equipo": "Price_Equipo1",
      "Fecha": "2023-09-30",
      "Prediccion": 451.73,
      "Metodo": "Naive: último precio mensual"
    },
    {
      "Equipo": "Price_Equipo2",
      "Fecha": "2023-09-30",
      "Prediccion": 955.35,
      "Metodo": "Naive: último precio mensual"
    }
  ],
  "metricas_disponibles": [],
  "adve

In [6]:
from openai import OpenAI

def conectar_openai():
    api_key = None
    try:
        from google.colab import userdata
        api_key = userdata.get('OPENAI_API_KEY')
    except (ImportError, KeyError, Exception):
        api_key = os.environ.get('OPENAI_API_KEY')
    if not api_key:
        raise RuntimeError('Configura OPENAI_API_KEY en Secretos de Colab o variable de entorno.')
    return OpenAI(api_key=api_key)


## 6. Función principal del agente

La API recibe **una pregunta y un contexto pequeño, estructurado y calculado localmente**. Pedimos responder solo con esos resultados, mencionar fecha y método y reconocer lo que se desconoce.

Esto no equivale a un agente autónomo con herramientas. Es un **asistente de IA generativa basado en contexto**, más adecuado como demostración junior sencilla.

Si se pregunta por 2026, la respuesta debe explicar que no tenemos información vigente. Si se pregunta por métricas de ML sin cargar `metricas.csv`, debe reconocer que no están disponibles.

In [7]:
INSTRUCCIONES = """Eres un asistente para una prueba de Data Science Junior sobre precios de equipos.
Responde en español, claramente y de forma breve. Usa SOLO los datos incluidos en CONTEXTO.
No inventes precios, fechas, métricas, presupuestos, modelos ni probabilidades.
La correlación no implica causalidad. Distingue observado, baseline naive y pronóstico de modelo.
Si te piden precios actuales en 2026 o pronósticos posteriores al horizonte disponible,
explica que no hay datos adecuados. Si falta una métrica, dilo.
En presupuestos, solo usa las cantidades hipotéticas y el cálculo ya suministrado.
Indica la fecha de las estimaciones y que se trata de demostración académica."""

def preguntar_agente(pregunta, cantidades=None, modelo='gpt-4o-mini'):
    if not isinstance(pregunta,str) or not pregunta.strip():
        raise ValueError('Escribe una pregunta.')
    cliente = conectar_openai()
    contexto_pregunta = generar_contexto(cantidades=cantidades)
    respuesta = cliente.responses.create(
        model=modelo,
        instructions=INSTRUCCIONES,
        input=f'CONTEXTO:\n{contexto_pregunta}\n\nPREGUNTA:\n{pregunta}'
    )
    return respuesta.output_text


## 7. Demostración y límites



In [9]:
print(preguntar_agente('¿Cuál fue el último precio observado del Equipo 1?'))

El último precio observado del Equipo 1 fue de 451.73.


In [10]:
print(preguntar_agente('¿Qué relación existe entre Z y el Equipo 2?'))

La correlación observada entre el Precio Z y el Precio del Equipo 2 es de 0.982. Esto indica una fuerte relación no causal, pero no implica que uno cause el comportamiento del otro.


In [11]:
print(preguntar_agente('¿Qué se estima para el siguiente mes disponible?'))

Para el siguiente mes disponible, se estima lo siguiente:

- **Price_Equipo1**: 451.73 (pronóstico basado en el método naive: último precio mensual)
- **Price_Equipo2**: 955.35 (pronóstico basado en el método naive: último precio mensual)

Estas estimaciones son para la fecha de **2023-09-30**.


In [12]:
print(preguntar_agente('¿Cuánto costarían 15 equipos 1 y 10 equipos 2?', cantidades={'Price_Equipo1':15,'Price_Equipo2':10}))

El costo de 15 equipos 1 y 10 equipos 2, según el presupuesto hipotético, sería:

- Precio estimado de 1 Equipo 1: 451.73
  - Subtotal para 15 Equipos 1: 15 * 451.73 = 6775.95

- Precio estimado de 1 Equipo 2: 955.35
  - Subtotal para 10 Equipos 2: 10 * 955.35 = 9553.50

**Total**: 6775.95 + 9553.50 = **16329.45**

Esta información corresponde a una demostración académica al 30 de septiembre de 2023.


In [13]:
print(preguntar_agente('¿Cuál es el precio vigente en octubre de 2026?'))

No se dispone de precios actuales de 2026. La información es hasta agosto de 2023 y se trata de una demostración académica.
